# Multi-Agent Workflows

A fully automated research pipeline built with the **OpenAI Agents SDK** (`openai-agents`): give it a question and it goes end to end — search the web, analyze, and write a polished executive report — with no manual steps.

- **`tavily_search`** — custom web-search tool (`https://api.tavily.com/search`, Bearer auth).
- **Researcher** (`gpt-5-mini`) — uses `tavily_search`, returns ≤ 5 factual bullet points.
- **Analyst** (`gpt-5-mini`) — no tools; extracts trends / risks / insights in 2 paragraphs.
- **Writer** (`gpt-5-mini`) — no tools; produces a `ResearchReport` (short summary, 500+ word Markdown report, 3–5 follow-up questions).
- **`manager_run(query)`** — chains Researcher → Analyst → Writer over a shared `SQLiteSession`.
- Pydantic models (`Report`, `ResearchReport`) carry data cleanly between the agents.

## 1. Setup

In [1]:
# Install dependencies (safe to re-run). Restart the kernel afterwards if this is a fresh venv.
%pip install -q openai-agents python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\ajaya\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os

from dotenv import load_dotenv

# Load OPENAI_API_KEY and TAVILY_API_KEY from the existing .env file.
load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY missing from .env"
print("API keys loaded ✓")

API keys loaded ✓


## 2. Tavily web search tool

The tool parameters are described with a `TypedDict`. `query` is required; `max_results` is optional and defaults to **2**. The core logic lives in a plain helper (`_run_tavily_search`) so it can be tested directly, while `tavily_search` is the `@function_tool`-decorated version that agents will call.

In [3]:
from typing import Any, NotRequired, TypedDict

import requests
from agents import function_tool

TAVILY_ENDPOINT = "https://api.tavily.com/search"
DEFAULT_MAX_RESULTS = 2


class TavilySearchInput(TypedDict):
    """Parameters for a Tavily web search."""

    query: str  # the search query
    max_results: NotRequired[int]  # number of results to return (default 2)


def _run_tavily_search(args: TavilySearchInput) -> dict[str, Any]:
    """Call the Tavily /search endpoint and return the parsed JSON response."""
    query = args["query"]
    max_results = args.get("max_results", DEFAULT_MAX_RESULTS)

    response = requests.post(
        TAVILY_ENDPOINT,
        headers={
            "Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}",
            "Content-Type": "application/json",
        },
        json={"query": query, "max_results": max_results},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


@function_tool
def tavily_search(args: TavilySearchInput) -> dict[str, Any]:
    """Search the web with Tavily for current information.

    Provide a `query` and optionally `max_results` (defaults to 2). Returns the
    raw Tavily JSON, including a `results` list of matching web pages.
    """
    return _run_tavily_search(args)

## 3. Quick test

Call the underlying helper directly (no agent required) to confirm the tool works end to end.

In [4]:
results = _run_tavily_search({"query": "OpenAI Agents SDK latest version", "max_results": 2})

print(f"Got {len(results.get('results', []))} result(s):\n")
for item in results.get("results", []):
    print(f"- {item['title']}\n  {item['url']}\n")

Got 2 result(s):

- Agents SDK from OpenAI! | Full Tutorial
  https://www.youtube.com/watch?v=35nxORG1mtg

- Release process/changelog - OpenAI Agents SDK
  https://openai.github.io/openai-agents-python/release



## 4. Shared data model

A single Pydantic `BaseModel` with a `summary` field is used as the output type for **both** agents, so the Researcher's findings pass straight into the Analyst without any manual reshaping.

In [5]:
from pydantic import BaseModel


class Report(BaseModel):
    """Shared output type so data flows cleanly between agents.

    The Researcher fills `summary` with up to 5 factual bullet points; the Analyst
    fills it with a 2-paragraph analysis of those notes.
    """

    summary: str

## 5. Specialized agents

Two single-skill agents, both on `gpt-5-mini` and both returning a `Report`:

- **Researcher** — has the `tavily_search` tool; gathers facts and returns ≤ 5 bullet points.
- **Analyst** — no tools; turns research notes into trends / risks / insights across 2 paragraphs.

Using the same `Report` output type lets the Researcher's output flow cleanly into the Analyst.

In [6]:
from agents import Agent

# Verified available via the OpenAI models docs; used exactly as specified (no substitution).
MODEL = "gpt-5-mini"

# Agent 1: searches the web and collects facts (no analysis).
researcher = Agent(
    name="Researcher",
    instructions=(
        "You are a web researcher. Use the tavily_search tool to gather real-time, "
        "verifiable facts about the user's topic. Collect concrete facts only — do NOT "
        "analyze or interpret. Return at most 5 concise bullet points in the `summary` "
        "field, each a single factual statement."
    ),
    model=MODEL,
    tools=[tavily_search],
    output_type=Report,
)

# Agent 2: receives research notes and analyzes them (no tools).
analyst = Agent(
    name="Analyst",
    instructions=(
        "You are a research analyst. You receive research notes and extract the key "
        "trends, risks, and insights. Write exactly 2 paragraphs in the `summary` field. "
        "Do not use any tools; reason only from the notes you are given."
    ),
    model=MODEL,
    output_type=Report,
)

## 6. Test the Researcher

Run the Researcher on the test query. It calls `tavily_search`, then returns at most 5 bullet points in the `summary` field of a `Report`. (`await` works at the top level inside a notebook.)

In [7]:
from agents import Runner

TEST_QUERY = "Why is Labubu so popular and what are the rarest Labubu collectibles?"

research_result = await Runner.run(researcher, TEST_QUERY)

print("=== Researcher output (max 5 bullets) ===\n")
print(research_result.final_output.summary)

=== Researcher output (max 5 bullets) ===

- Labubu is a designer toy character created by Hong Kong artist Kasing Lung and produced/distributed globally by Pop Mart.
- Labubu figures and plushies are commonly sold in Pop Mart "blind box" releases.
- Some "secret" Labubu variants have reported drop rates as rare as 1 in 72 or 1 in 144 boxes.
- Certain Labubu collectibles and limited editions have resold for six-figure sums.
- Labubu has become a popular fashion accessory among young adults, often clipped to bags or belt loops.


## 7. Report model

The Writer needs a richer output than the shared `Report`, so it gets its own `ResearchReport` model: a short summary, a full Markdown report (500+ words), and a list of follow-up questions.

In [8]:
class ResearchReport(BaseModel):
    """Final, polished report produced by the Writer agent."""

    short_summary: str  # 2-3 sentence executive summary
    markdown_report: str  # full report in Markdown, 500+ words
    follow_up_questions: list[str]  # 3-5 suggested follow-up questions

## 8. Writer agent

The **Writer** (`gpt-5-mini`, no tools) turns the research and analysis into a finished, structured report via the `ResearchReport` output type.

In [9]:
# Agent 3: turns research + analysis into a polished executive report (no tools).
writer = Agent(
    name="Writer",
    instructions=(
        "You are a senior research writer. Using the user's original question, the "
        "research notes, and the analysis, produce a polished executive report:\n"
        "- short_summary: 2-3 sentences capturing the headline findings.\n"
        "- markdown_report: a well-structured Markdown report of AT LEAST 500 words, "
        "with headings, covering background, detailed analysis, and implications.\n"
        "- follow_up_questions: 3 to 5 insightful questions for further research.\n"
        "Do not use any tools; write only from the material provided."
    ),
    model=MODEL,
    output_type=ResearchReport,
)

## 9. End-to-end pipeline

`manager_run(user_query)` chains **Researcher → Analyst → Writer**. A single `SQLiteSession` is shared across all three runs so they build on a common memory of the conversation:

1. **Researcher** gets the raw query → research summary.
2. **Analyst** gets the research summary → analysis.
3. **Writer** gets query + research + analysis → final `ResearchReport`.

In [10]:
from uuid import uuid4

from agents import SQLiteSession


async def manager_run(user_query: str) -> ResearchReport:
    """Run the full Researcher -> Analyst -> Writer pipeline end to end.

    All three agents share one SQLiteSession, so each step's turns are stored and
    available as conversation memory to the next agent. Returns the Writer's report.
    """
    # One shared session id per run gives all agents common memory.
    session = SQLiteSession(f"research-{uuid4().hex[:8]}")

    # 1. Researcher: gather facts from the web.
    research = await Runner.run(researcher, user_query, session=session)
    research_summary = research.final_output.summary

    # 2. Analyst: extract trends / risks / insights from the research.
    analysis = await Runner.run(
        analyst,
        f"Research notes for the question '{user_query}':\n\n{research_summary}",
        session=session,
    )
    analysis_summary = analysis.final_output.summary

    # 3. Writer: compose the polished report from query + research + analysis.
    writer_input = (
        f"Original question: {user_query}\n\n"
        f"Research notes:\n{research_summary}\n\n"
        f"Analysis:\n{analysis_summary}"
    )
    report = await Runner.run(writer, writer_input, session=session)
    return report.final_output

## 10. Run the full pipeline

One call — `manager_run(query)` — goes from raw question to finished report: summary, full Markdown report, and follow-up questions. No manual steps.

In [11]:
from IPython.display import Markdown, display

PIPELINE_QUERY = "What is the public sentiment and expert reviews about the Tesla Cybertruck?"

report = await manager_run(PIPELINE_QUERY)

print("=== SHORT SUMMARY ===\n")
print(report.short_summary)

print("\n=== FOLLOW-UP QUESTIONS ===\n")
for i, question in enumerate(report.follow_up_questions, 1):
    print(f"{i}. {question}")

print("\n=== FULL REPORT ===\n")
display(Markdown(report.markdown_report))

=== SHORT SUMMARY ===

Public opinion on the Cybertruck is mixed: small-sample owner reviews (Kelley Blue Book) are strongly positive, while broader public sentiment toward Tesla and Elon Musk is notably negative. Expert coverage praises the Cybertruck’s technical performance and range but also highlights design controversy, value questions (e.g., cost of add-on range extender), and signs of cooling demand/discounting that create volume risk.

=== FOLLOW-UP QUESTIONS ===

1. What do larger, representative owner-satisfaction surveys (hundreds to thousands of owners) say about long-term reliability, real-world range, and ownership costs for the Cybertruck?
2. How have Cybertruck sales and order cancellations trended month-to-month since launch, and how do discounting strategies affect resale values and brand perception?
3. What are independent real-world tests (towing, payload, cold-weather range) saying about the Cybertruck’s practical utility compared with rivals like the Ford F-150 Li

# Tesla Cybertruck: Public Sentiment and Expert Reviews

## Background
The Tesla Cybertruck launched as one of the most polarizing vehicles in recent automotive history — its angular, stainless-steel exterior and Tesla’s headline-grabbing marketing split observers between enthusiastic early adopters and skeptical mainstream buyers. Since launch, reviewers and the market have focused on three areas: owner satisfaction, expert technical and driving appraisals, and market uptake/price signals.

## Public Sentiment: Owners vs. General Public
- Owner feedback (small sample): Kelley Blue Book’s 2025 consumer reviews show very high satisfaction among a small group of owners: among seven reviewers of the 2025 Cybertruck, roughly 85% recommended the vehicle and six of seven rated it five out of five stars. These responses suggest strong enthusiasm from early owners — likely people who sought the vehicle out and value its novelty and performance.

- Broader public sentiment: A broader picture of brand sentiment differs substantially. A CNBC All-America Economic survey (April 2025) found approximately 47% of Americans held a negative view of Tesla, and around half had a negative view of CEO Elon Musk. This indicates a sizable portion of the general population is predisposed against the company — a headwind for broad mainstream adoption that can affect purchase intent even if the product itself is well reviewed by owners.

Interpretation: Early adopters can be intensely positive while general sentiment remains wary. KBB’s owner sample is small and likely biased toward enthusiasts; larger, representative owner-satisfaction surveys would be needed to generalize.

## Expert Reviews and Technical Appraisal
- Performance and range: Automotive press coverage (Car and Driver, MotorTrend) emphasizes the Cybertruck’s strong technical specs. All trims reportedly exceed roughly 300 miles of range; the high-end tri-motor “Cyberbeast” claims a 0–60 mph time around 2.6 seconds, positioning it among the quickest trucks.

- Features and product evolution: MotorTrend and other outlets note Tesla’s ongoing software and hardware strategy — over-the-air updates and planned add-ons such as a bed-mounted range-extending battery pack (announced for mid-2025) that Tesla says will add about 120 miles at an estimated cost near $16,000.

- Tone of expert coverage: Reviews tend to be mixed-to-positive on capability (acceleration, range, novel features like steer-by-wire) but tempered by criticisms: controversial styling, practical trade-offs, reliability and recall histories for Tesla models, and questions about true towing/utility under real-world conditions.

## Market Signals: Demand and Pricing
Industry reporting summarized in automotive coverage has indicated signs of cooling demand: some outlets reported that Tesla applied discounts as sales softened and industry analysts questioned whether the Cybertruck would reach aggressive volume goals. Those signals matter because discounts can change perception (from exclusive/new to clearance/overstock), affect resale values, and alter the economics for both Tesla and prospective buyers.

## Analysis: Reconciling the Divide
- Positive technical reviews vs. public skepticism: Experts largely acknowledge the Cybertruck’s performance and capabilities, while the broader public’s view of Tesla and its CEO can dampen mainstream enthusiasm. Early owners’ strong satisfaction coexists with general brand skepticism.

- Value perception questions: The proposed bed-mounted range extender (120 miles for ~$16,000) demonstrates Tesla is addressing range anxiety but raises questions about cost-effectiveness. If high-cost add-ons become necessary to meet buyer expectations, the Cybertruck’s value proposition vs. competitors could be weakened.

- Demand uncertainty: Discounting and reports of weaker demand imply the vehicle’s niche appeal may limit volume. Early adopter positivity may not translate to mass-market sales without improved mainstream perceptions or adjusted pricing/feature bundles.

## Implications
- For Tesla: The company must manage brand perception and communicate value clearly. Continued OTA improvements and tangible feature additions (e.g., verified range extender rollouts) help, but price strategy and reliability/ownership experience will determine broader adoption.

- For consumers: Enthusiasts can expect a high-performance, distinctive truck with strong specs; mainstream buyers should weigh design preference, resale risk, and the possibility of higher total cost if desired range/features require pricey options.

- For competitors: A technically strong but niche Cybertruck opens opportunities for more conventional EV truck makers to win mainstream pickup buyers by emphasizing familiar design, proven utility, and clearer value.

## Uncertainties and Risks
- Small owner-sample bias in satisfaction reporting.
- Public sentiment tied to the corporate figurehead (Musk) can shift quickly and impact sales independent of vehicle quality.
- Real-world durability, towing, and long-term ownership costs remain to be proven at scale.

## Conclusion
Expert reviewers generally praise the Cybertruck’s performance and range while noting design and practical trade-offs. Early owners report high satisfaction, but broader public sentiment toward Tesla and Elon Musk is markedly mixed-to-negative, and industry signals of cooling demand and discounting create execution risk. The Cybertruck’s future acceptance will depend on Tesla’s ability to broaden its appeal, justify add-on costs, and sustain positive ownership experiences at scale.
